# MarginOps - Site Trading Data Cleaning 

## Objective 

Clean and validate site-level trading data before analysis in SQL, Excel and Power BI, covering revenue, cost, margin and other performance KPIs.

## Business context

The dataset contains site, date, shift, covers, food revenue, wet revenue, promotions, discounts, service charge, tips, food COGS, wet COGS, wastage cost and forecast revenue. 

## Cleaning principles

- Raw data is never overwritten. 
- Ambiguous values are preserved where possible.
- Duplicate records are investigated before removal. 
- Imputed values are labelled. 
- Exclusions are documented.

# Importing libraries and configurations 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 
import seaborn as sns 

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "cleaning":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "site_trading.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "cleaned" / "site_trading_cleaned.csv"

# Loading raw data


In [ ]:
df_raw = pd.read_csv(RAW_PATH)

df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}") 
print(f"Columns: {df.shape[1]:,}")

df.head()
df.tail()
df.info()
df.describe(include="all").T
df.isna().sum().sort_values(ascending=False)
df.nunique().sort_values() 

## Initial observations

- The dataset contains 7,500 rows and 17 columns.
- The date range is from 08/2024 to 08/2026.
- The primary operating dimensions are site, date and shift.
- Missing values are concentrated in forecast revenue.
- Potential data-quality concerns include missing data in covers, both revenue, food cost, wet cost of sales, wastage and forecast revenue. 
- Need to confirm the integrity of is closed on certain sites.  

# Data dictionary

| Column | Meaning | Expected type | Used in analysis | 
|--------|---------|---------------|------------------|
| site | Trading location | String | Yes |
| date | Trading date | Datetime | Yes |
| day_of_week | Day of the week | String | Yes |
| shift | Lunch or Dinner | String | Yes |
| covers | Numbers of customers | Float | Yes |
| food_revenue | Food sales | Float | Yes | 
| wet_revenue | Drinks sales | Float | Yes | 
| promotions | Promotions from a site | Float | Yes | 
| discounts | Any discounts put on orders | Float | Yes | 
| service_charge | Split between the whole team | Float | Yes | 
| tips | Any money made from Front of House staff | Float | Yes | 
| food_cost | Food cost of goods sold | Float | Yes |
| wet_cost_of_sales | Drink cost of goods sold | Float | Yes |
| wastage_cost | The cost of any inventory that has been discarded | Float | Yes | 
| forecast_revenue | The projected revenue of the site | Float | Yes | 
| is_closed | Whether a site was closed or not | Boolean | Possibly |
| notes | Any qualitative comments | String | Possibly | 

# Standardising columns and data types 


In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [ ]:
df["site"] = df["site"].str.strip().str.lower()
df["shift"] = df["shift"].str.strip().str.lower()

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print(df["date"])

In [ ]:
numeric_columns = [
    "covers",
    "wet_revenue", 
    "food_revenue", 
    "promotions",
    "discounts", 
    "service_charge", 
    "tips", 
    "food_cost",
    "wet_cost_of_sales", 
    "wastage_cost", 
    "forecast_revenue", 
]

for column in numeric_columns: 
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Data quality investigation 

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count}")

In [ ]:
key_columns = ["site", "date", "shift"]

key_duplicates = (
    df[df.duplicated(key_columns, keep=False)]
    .sort_values(key_columns)
)

key_duplicates

In [ ]:
print(df['date'].head(10))
print(df['date'].min())
print(df['date'].max())

In [ ]:
df["derived_day"] = df["date"].dt.day_name().str.lower()

df["day_mismatch"] = (
    df["day_of_week"].str.strip().str.lower()
    != df["derived_day"]
)

df["day_mismatch"].sum()

In [ ]:
financial_columns = [
    "food_revenue",
    "wet_revenue",
    "promotions",
    "discounts",
    "service_charge",
    "tips",
    "food_cost",
    "wet_cost_of_sales",
    "wastage_cost",
    "forecast_revenue"
]

negative_summary = {
    column: (df[column] < 0).sum()
    for column in financial_columns
}

pd.Series(negative_summary).sort_values(ascending=False)

# Cleaning and observation 


In [ ]:
df["is_exact_duplicate"] = df.duplicated(keep="first")

df["revenue_missing"] = (
    df[["food_revenue", "wet_revenue"]]
    .isna()
    .any(axis=1)
)

df["cost_cogs_missing"] = (
    df[["food_cost", "wet_cost_of_sales"]]
    .isna()
    .any(axis=1)
)

df["covers_missing"] = df["covers"].isna()

df["date_parsed"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

In [ ]:
original_columns = df_raw.columns.drop("notes") # just for the chart

missing_percentage = (
    df_raw[original_columns] 
    .isna()
    .mean()
    .mul(100) 
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=missing_percentage.values,
    y=missing_percentage.index,
    color="steelblue"
)

plt.title("Missing Values by Field")
plt.xlabel("Missing values (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Interpretation 

The notes field was excluded from this chart because it contains mostly
qualitative blanks and would dominate the visual. Among the operational and
financial fields shown, forecast revenue has the highest missingness,
followed by wet revenue and other trading fields. No values were replaced.

In [ ]:
plot_df = df.copy()

# Excluding confirmed duplicate records only from this visual
plot_df = plot_df[
    ~plot_df["is_exact_duplicate"]
].copy()

# Keeping rows where both revenue fields are present 
plot_df["actual_revenue"] = plot_df[
    ["food_revenue", "wet_revenue"]
].sum(axis=1, min_count=2)

plot_df["month"] = (
    plot_df["date_parsed"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Fixing site names for this visualisation

plot_df["site_display"] = (
    plot_df["site"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.title()
)

monthly_revenue = (
    plot_df
    .dropna(subset=["actual_revenue", "month"])
    .groupby(["month", "site_display"], as_index=False)["actual_revenue"]
    .sum()
)

In [ ]:
plt.figure(figsize=(14, 7))

sns.lineplot(
    data= monthly_revenue,
    x="month",
    y="actual_revenue",
    hue="site_display",
    marker="o"
)

plt.title("Monthly Actual Revenue by Site")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.legend(title="Site", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Exploratory note

This visual excludes rows flagged as exact duplicates and rows where either
food revenue or wet revenue is missing. These exclusions apply only to this
chart; no records were removed from the validated dataset.

The chart provides an initial view of monthly revenue patterns by site.
Formal KPI definitions and business conclusions will be developed in Power BI.

## Cleaning decisions

No original financial, operational or qualitative values were overwritten. 

Instead: 
- Missing values were retained;
- Negative values were retained; 
- Duplicate records were identified and flagged; no rows were removed;
- Missing revenue was flagged but not converted to zero;
- Dates were converted to datetime format and validated against the recorded day;
- Site and shift values were standardised in the working dataframe, while the raw values remained preserved in `df_raw`;
- Exclusions will be applied only in the relevant analysis stage.

# Auditing

In [ ]:
df["date_parsed"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df["derived_day"] = (
    df["date_parsed"]
    .dt.day_name()
    .str.lower()
)

df["day_mismatch"] = (
    df["day_of_week"]
    .astype("string")
    .str.strip()
    .str.lower()
    != df["derived_day"]
)

df["is_key_duplicate"] = df.duplicated(
    subset=["site", "date", "shift"],
    keep=False
)

negative_rows = (
    df[financial_columns]
    .lt(0)
    .any(axis=1)
    .sum()
)

audit_summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "validated_rows",
        "exact_duplicate_rows_flagged",
        "key_duplicate_rows_flagged",
        "missing_revenue_rows",
        "missing_covers_rows",
        "missing_cost_of_cogs_rows",
        "invalid_dates",
        "day_mismatch_rows",
        "negative_financial_rows_flagged",
        "rows_removed"
    ],
    "value": [
        len(df_raw),
        len(df),
        int(df["is_exact_duplicate"].sum()),
        int(df["is_key_duplicate"].sum()),
        int(df["revenue_missing"].sum()),
        int(df["covers_missing"].sum()),
        int(df["cost_cogs_missing"].sum()),
        int(df["date"].isna().sum()),
        int(df["day_mismatch"].sum()),
        int(negative_rows),
        0
    ]
})

display(audit_summary)

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)

# Conclusion and handoff

The site-trading dataset has completed its cleaning and validation stage.

The raw dataset was preserved separately and was not overwritten. Site and
shift values were standardised in the working dataframe, while the original
raw values remained preserved in `df_raw`. Dates were converted to datetime
format and validated against the recorded day of the week. No day-of-week
mismatches were identified.

Missing values and negative financial values were retained and flagged rather
than changed. Duplicate records were investigated, with later exact duplicate
copies identified for exclusion from exploratory analysis while remaining
preserved in the validated dataset.

The validated site-trading CSV has been exported and is ready for SQL
analysis, Excel reconciliation and Power BI visualisation. KPI definitions
and analytical exclusions will be documented during the next stage.